In [14]:
import xarray
import numpy
import matplotlib.pyplot as plt
from matplotlib.tri import Triangulation
import cartopy.crs as ccrs
from cartopy.io.shapereader import Reader
from cartopy.feature import ShapelyFeature
from metpy.units import units
import pyart
import numpy as np
from metpy.calc import temperature_from_potential_temperature, virtual_temperature, mixing_ratio_from_specific_humidity
from metpy.calc import dewpoint_from_specific_humidity, relative_humidity_from_specific_humidity, wind_speed
from datetime import datetime, timedelta
from wrf import getvar
from netCDF4 import Dataset
#import satColorMaps as sCM
import s3fs
import json
import urllib.request
import numexpr as ne
import xarray as xr
import matplotlib
from metpy.interpolate import log_interpolate_1d

In [6]:
hour_t = 10
hour_s = '10'

dt = datetime(2022,9,15,hour_t)
#Load in WRF data from both runs
def read_wrfregrid(exper):
    #wrfout = Dataset('/glade/derecho/scratch/mawilson/20210604_newcase/'+exper+'/wrfout_d01_2022-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':00:00')
    wrfout = Dataset('/glade/derecho/scratch/mawilson/mpas_tests/ohio_mesh/regrid_nature'+exper+'_'+str(dt.isoformat()[11:13])+':00:00.nc')
    
    #wrfout = Dataset('/glade/derecho/scratch/mawilson/20210604_newcase/'+exper+'/wrfout_d02_2022-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':00:00')
    #wrfout = Dataset('/glade/derecho/scratch/mawilson/20210604_newcase/'+exper+'/wrfout_d02_2021-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':00:00')
    
    U10 = wrfout.variables['u10m']
    V10 = wrfout.variables['v10m']
    T2 = np.asarray(wrfout.variables['t2m'])*units('K')
    T2F = T2 .to('degF')
    Q2 = np.asarray(wrfout.variables['q2m'])
    RH2 = np.asarray(wrfout.variables['rh2m'])
    SPD10 = wind_speed(np.asarray(U10)*units('m/s'), np.asarray(V10)*units('m/s'))
    cloud=wrfout.variables['cloud']
    Q3 = np.asarray(wrfout.variables['q3'])
    T3 = np.asarray(wrfout.variables['t'])
    P3 = np.asarray(wrfout.variables['p3'])
    #print(wrfout.variables)

    return cloud, RH2, T2F, SPD10, Q2, Q3, T3, P3

cloudn2, RH2n2, T2n2, SPD10n2, Q2m2, Q3, T3, P3 = read_wrfregrid('09-15')

In [21]:
for i in range(P3.shape[1]):
    for j in range(P3.shape[2]):
        if np.max(P3[:,i,j]) > 0:
            T2_a = log_interpolate_1d([990,975,950,925,900,875,850,825,800,750,700]*units('hPa'), P3[:,i,j]*units('hPa'), T3[:,i,j]*units('degC'))
            print(T2_a)

[18.601690329957037 19.650573841305636 19.033678785116944 18.2094318177587 17.264971902783973 16.48861893647042 15.598084233758332 14.334509047181118 12.609973482742586 9.432310035184036 5.897070759555845] degree_Celsius
[17.292249097273036 20.202218685794726 19.47374338205598 18.437977733326157 17.61853428741858 17.1716160371168 16.174153846207563 14.685963277825763 12.906511922634913 9.507725203842728 7.128957530872067] degree_Celsius
[17.875877091951764 20.493147896923766 19.659585739661047 18.24532161039953 17.301069769037376 16.75850659674565 15.811724005992515 14.399874103070811 12.704548175510089 9.58057211543076 7.257174717759079] degree_Celsius
[17.12999877673053 20.266873761881225 19.38529332958259 17.887005203553745 17.206572989807963 16.83743529944016 15.847632853245056 14.357911586972461 12.593143708745917 9.714621149772437 7.1767927211824185] degree_Celsius
[17.20114284417829 20.27638738179606 19.27203304463928 17.769112501361924 17.26014648963358 16.911137474377874 15.85

/glade/derecho/scratch/mawilson/tmp/ipykernel_33597/1552801145.py:4: UserWarning: Interpolation point out of data bounds encountered
  T2_a = log_interpolate_1d([990,975,950,925,900,875,850,825,800,750,700]*units('hPa'), P3[:,i,j]*units('hPa'), T3[:,i,j]*units('degC'))


[18.9529977925017 20.431614234284414 19.555258901245377 18.50143006315731 17.175450590720114 16.395810408326366 15.579569134179849 14.218277669806914 12.517501207880859 9.777551881950243 7.185498483220379] degree_Celsius
[18.296486394967157 20.35995645449093 19.37963137482214 18.366077366848565 17.138690104173907 16.169316185077225 15.47808434361372 14.251753249405088 12.593279397393102 9.799508716799128 7.1487400569912065] degree_Celsius
[nan 20.389503622320145 19.3229260060253 18.352532720825707 17.143178122306878 16.06764972684366 15.383437475065746 14.180342018435708 12.591377647404295 9.720155581573866 7.074551646629171] degree_Celsius
[17.282239324083367 19.971664585836972 19.21433180914573 18.202865664210474 17.10679465782353 16.043990525498902 15.363534673026823 14.2433581275374 12.661917579555114 9.754256809003591 6.932363581548367] degree_Celsius
[17.965176803632104 19.95591400554098 19.050208765566552 18.012489109902198 17.045796069760893 16.027449937191257 15.35668351733435